# Paired agent run: v4_semdag vs v5_vulnwalk vs v6_impact (Gemma 4 E4B, CUDA, L4x4)

Runs the **real** harness (`swegemma.Evaluator`) with the **real** CUDA E4B checkpoint, one task at a time, for three submissions on the same tasks. It writes per-task results, paired tables, per-task timings and a projected 120-task wall time against the 12 h limit.

**Settings to attach before running**
- Accelerator: GPU L4 x4. Internet: off.
- Inputs: the competition `gemma-4-developer-agent`, the dataset `metric/gemma-4-developer-agent-wheelhouse`, and the model `google/gemma-4` (variant `gemma-4-e4b-it-qat-w4a16-ct`, version 2).

Default run: 30 tasks (15 locked + 15 held-out), all three versions on each. The submissions are embedded in cell 3, so no upload is needed. All run under **the same budgets** (cell 2), not their shipped `eval_config.yaml`, so the comparison is not confounded by v4's 8-minute setting.

**Path fix for v4 and v5.** This notebook uses the subprocess sandbox, like the official one. There the repository is not at `/workspace`, and v4 and v5 read that literal path inside their skill script, so their localizer returns nothing. With `PATCH_WORKSPACE_PATH = True` (default) the notebook applies the smallest fix to its own copies of v4 and v5, so the comparison measures the methods and not a dead tool. Set it to `False` to run v4 and v5 exactly as shipped. v6 has the fix built in.

Not tested before shipping: the vLLM start, the `Evaluator` loop and the timing math were written from the official getting-started notebook but could not be run without a GPU. Cells 3 to 4 were run offline against the real validator, and the path fix was run in the harness's real subprocess sandbox.


In [ ]:
import glob, importlib, logging, os, shutil, subprocess, sys
from pathlib import Path

for k, v in {
    'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1', 'TRANSFORMERS_OFFLINE': '1',
    'VLLM_NO_USAGE_STATS': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
    'NCCL_DEBUG': 'WARN', 'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'LITELLM_TELEMETRY': 'False',
    'LITELLM_LOG': 'WARNING', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true',
}.items():
    os.environ[k] = v
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')

for name in ('LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm', 'httpx', 'httpcore', 'openai', 'urllib3', 'opentelemetry'):
    lg = logging.getLogger(name); lg.setLevel(logging.WARNING)

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass

# Restore '+cu128' wheel filenames that Kaggle dataset uploads strip (same as the official notebook).
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp_whl / name
    if not target.exists():
        try: os.symlink(w, target)
        except OSError: shutil.copyfile(w, target)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('done')

In [ ]:
# ---- Configuration -------------------------------------------------------------------------
MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
VERSIONS = ['v4_semdag', 'v5_vulnwalk', 'v6_impact']   # all are embedded below; drop one to save GPU time
PATCH_WORKSPACE_PATH = True                  # fix v4/v5 reading the literal /workspace (see the note above)
RUN_SETS = ['locked15', 'heldout15']         # 15 locked + 15 held-out = 30 tasks, every version on each
# Same budgets for every version (what the scorer reads from eval_config.yaml):
BUDGET = dict(timeout_seconds=120, max_tool_calls=40, max_time_minutes=5, max_turns=60)
HOURS_LIMIT = 12.0
STARTUP_OVERHEAD_H = 0.5                     # vLLM start-up plus container setup allowance for the projection

In [ ]:
import base64, io, zipfile, re, yaml
from pathlib import Path

WORKING = Path('/kaggle/working'); WORKING.mkdir(parents=True, exist_ok=True)
SUBS = WORKING / 'subs'
EMBED_B64 = "..."

shutil.rmtree(SUBS, ignore_errors=True); SUBS.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBED_B64))) as z:
    z.extractall(SUBS)
def apply_workspace_pathfix(sub_dir, wsroot_source):
    """v4 and v5 read the literal path /workspace inside their skill script. Under the subprocess sandbox
    that the Kaggle notebook uses, that directory does not exist, so their localizer returns nothing.
    This applies the smallest fix to a COPY of the submission: resolve the repository from the harness
    wrapper's location (wsroot.py, taken from v6). The frozen v4 and v5 folders are not changed."""
    from pathlib import Path
    scripts = Path(sub_dir) / 'skills' / 'semdag' / 'scripts'
    (scripts / 'wsroot.py').write_text(wsroot_source, encoding='utf-8')
    p = scripts / 'repo_eval.py'
    s = p.read_text(encoding='utf-8')
    a = 'os.environ.setdefault("SEMDAG_CACHE", "/tmp/semdag-cache")'
    b = 'import wsroot\nos.environ["SEMDAG_CACHE"] = wsroot.cache_root()'
    c = '    a = ap.parse_args()\n'
    d = c + '    a.repo_dir = wsroot.workspace_root(a.repo_dir)\n'
    assert s.count(a) == 1 and s.count(c) == 1, 'unexpected repo_eval.py'
    p.write_text(s.replace(a, b).replace(c, d), encoding='utf-8')

WSROOT_SOURCE = (SUBS / 'v6_impact' / 'skills' / 'semdag' / 'scripts' / 'wsroot.py').read_text(encoding='utf-8')
for v in VERSIONS:
    d = SUBS / v
    assert d.is_dir(), f'missing embedded submission {v}'
    if PATCH_WORKSPACE_PATH and v in ('v4_semdag', 'v5_vulnwalk'):
        apply_workspace_pathfix(d, WSROOT_SOURCE)
        print(v, 'workspace path fix applied to the notebook copy')
    # identical budgets for all versions
    (d / 'eval_config.yaml').write_text(yaml.safe_dump({'evaluation': BUDGET}), encoding='utf-8')
    print(v, sorted(p.relative_to(d).as_posix() for p in d.rglob('*') if p.is_file())[:3], '...')

In [ ]:
from adk_submission import validate_directory
from adk_submission.discovery import discover_declared_models
from swegemma.config import build_submission_limits

limits, gen_constraints = build_submission_limits()
for v in VERSIONS:
    d = SUBS / v
    for dirpath, dirnames, filenames in os.walk(d, followlinks=False):
        for n in (*dirnames, *filenames):
            assert not (Path(dirpath) / n).is_symlink(), f'symlink in {v}: {n}'
    validate_directory(d, limits=limits)
    models_declared = discover_declared_models(d)
    assert models_declared == {MODEL_NAME}, f'{v}: declared models {sorted(models_declared)}'
    print(f'{v}: valid, single model {MODEL_NAME}')

In [ ]:
import random
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
by_id = {t.instance_id: t for t in tasks}
all_ids = sorted(by_id)

LOCKED15 = ['fastapi_14851', 'fastapi_13537', 'fastapi_14482', 'rich_3278', 'fastapi_12942', 'rich_3934', 'fastapi_14301',
            'fastapi_14372', 'fastapi_14258', 'rich_3472', 'fastapi_14262', 'fastapi_14455', 'requests_7433', 'fastapi_13920',
            'requests_7328']
# Integrity check: the locked sample must equal the pre-registered draw.
assert random.Random(20261113).sample(all_ids, 15) == LOCKED15, 'locked sample does not match seed 20261113 over the public ids'
HELDOUT15 = random.Random(20261114).sample([i for i in all_ids if i not in set(LOCKED15)], 15)

SETS = {'locked15': LOCKED15, 'heldout15': HELDOUT15}
RUN = [(s, i) for s in RUN_SETS for i in SETS[s]]
print(len(tasks), 'tasks;', len(RUN), 'to run')

In [ ]:
import torch, litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2')   # CUDA checkpoint, not MLX
assert MODEL_PATH.exists(), f'model weights not found at {MODEL_PATH}'

# No adapters in either submission (base E4B).
adapters = discover_adapters(str(SUBS / VERSIONS[0]), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
assert not (getattr(adapters, 'adapters', None) or {}), 'unexpected adapters'

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=False, max_loras=1, max_lora_rank=128,
    scheduler_reserve_full_isl=False, tensor_parallel_size=tp_size, startup_timeout=60 * 20, extra_args=(),
)
t_start = __import__('time').perf_counter()
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
VLLM_STARTUP_S = __import__('time').perf_counter() - t_start
print(f'vLLM up on {server_instance.base_url} (tp={tp_size}) in {VLLM_STARTUP_S:.0f}s')
models = server_instance.create_model_registry(aliases=[MODEL_NAME], model_prefix='openai/', api_key=os.environ['LOCAL_API_KEY'])

In [ ]:
import asyncio, concurrent.futures, json, time
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator

def run_sync(fn, *a, **kw):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn(*a, **kw))).result()
    return asyncio.run(fn(*a, **kw))

GRAPH_DIR = str(DATA_DIR / 'graphs'); EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

def make_evaluator(version):
    cfg = EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=WORKING / 'results' / version,
        submission_dir=SUBS / version, models=models, sandbox='subprocess',
        timeout_seconds=BUDGET['timeout_seconds'], verification_timeout_seconds=300,
        max_time_minutes=float(BUDGET['max_time_minutes']), max_tool_calls=BUDGET['max_tool_calls'],
        max_turns=BUDGET['max_turns'], limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, wheels_dir=DATA_DIR / 'wheels', verbose=False,
    )
    return Evaluator(cfg)

evaluators = {v: make_evaluator(v) for v in VERSIONS}
OUT = WORKING / 'paired'; (OUT / 'patches').mkdir(parents=True, exist_ok=True)
RESULTS = OUT / 'results.jsonl'
done = set()
if RESULTS.exists():                                   # resume after a restart
    for line in RESULTS.read_text().splitlines():
        r = json.loads(line); done.add((r['version'], r['task']))

# Task-major order: every version runs on a task before the next task starts, so a partial run stays paired.
n_total = len(RUN) * len(VERSIONS); k = 0
for set_name, tid in RUN:
    for v in VERSIONS:
        k += 1
        if (v, tid) in done: continue
        t0 = time.perf_counter()
        try:
            res = run_sync(evaluators[v].evaluate_task, task=by_id[tid], task_index=k, total_tasks=n_total)
            row = dict(version=v, set=set_name, task=tid, resolved=bool(res.resolved), exit_code=res.test_exit_code,
                       patch_chars=len((res.agent_patch or '').strip()), tool_calls=getattr(res, 'tool_calls', None),
                       duration_s=round(res.duration_seconds, 1), wall_s=round(time.perf_counter() - t0, 1),
                       error=(res.error or '')[:300])
            if (res.agent_patch or '').strip():
                (OUT / 'patches' / f'{v}__{tid}.diff').write_text(res.agent_patch, encoding='utf-8')
        except Exception as e:                          # a crash scores as unresolved but is recorded
            row = dict(version=v, set=set_name, task=tid, resolved=False, exit_code=None, patch_chars=0, tool_calls=None,
                       duration_s=None, wall_s=round(time.perf_counter() - t0, 1), error=f'{type(e).__name__}: {e}'[:300])
        with open(RESULTS, 'a') as f: f.write(json.dumps(row) + '\n')
        print(f"[{k}/{n_total}] {v:12s} {tid:16s} resolved={row['resolved']} patch={row['patch_chars']} wall={row['wall_s']}s")

In [ ]:
import pandas as pd
df = pd.DataFrame([json.loads(l) for l in RESULTS.read_text().splitlines()])
df.to_csv(OUT / 'results.csv', index=False)

summary = {'model': MODEL_NAME, 'budget': BUDGET, 'vllm_startup_s': globals().get('VLLM_STARTUP_S'), 'versions': {}}
for v in VERSIONS:
    d = df[df.version == v]
    mean_s = float(d.wall_s.mean()) if len(d) else None
    summary['versions'][v] = dict(
        tasks=int(len(d)), resolved=int(d.resolved.sum()), empty_patches=int((d.patch_chars == 0).sum()),
        errors=int((d.error != '').sum()), mean_wall_s=mean_s, median_wall_s=float(d.wall_s.median()) if len(d) else None,
        max_wall_s=float(d.wall_s.max()) if len(d) else None,
        projected_120_task_hours=(None if mean_s is None else round(mean_s * 120 / 3600 + STARTUP_OVERHEAD_H, 2)),
        fits_12h=(None if mean_s is None else bool(mean_s * 120 / 3600 + STARTUP_OVERHEAD_H <= HOURS_LIMIT)),
    )

# Paired tables: for every pair of versions, the tasks where both finished.
import itertools
summary['paired'] = []
for a, b in itertools.combinations(VERSIONS, 2):
    piv = df[df.version.isin([a, b])].pivot_table(index='task', columns='version', values='resolved', aggfunc='first').dropna()
    if len(piv) and a in piv and b in piv:
        summary['paired'].append(dict(first=a, second=b, tasks=int(len(piv)),
                                      both=int(((piv[a] == True) & (piv[b] == True)).sum()),
                                      only_first=int(((piv[a] == True) & (piv[b] == False)).sum()),
                                      only_second=int(((piv[a] == False) & (piv[b] == True)).sum()),
                                      neither=int(((piv[a] == False) & (piv[b] == False)).sum())))
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
display(df.sort_values(['task', 'version']))

In [ ]:
# Package each version as a Kaggle submission zip (same checks as the official notebook). Nothing is submitted.
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES
(WORKING / 'submissions').mkdir(exist_ok=True)
for v in VERSIONS:
    z = Path(shutil.make_archive(str(WORKING / 'submissions' / v), 'zip', root_dir=SUBS / v))
    with zipfile.ZipFile(z) as zf:
        infos = zf.infolist()
        assert sum(i.file_size for i in infos) <= MAX_SUBMISSION_SIZE_BYTES
        for i in infos:
            if not i.is_dir():
                assert Path(i.filename).suffix.lower() in ALLOWED_SUBMISSION_EXTENSIONS, i.filename
        assert 'agent.yaml' in zf.namelist()
    print('packaged', z, f'{z.stat().st_size/1024:.1f} KiB')
if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()